# 🌽 CornAI Model Retraining Notebook
Notebook ini digunakan untuk melatih ulang model YOLOv11 secara interaktif menggunakan **VS Code**.

### Alur Retraining:
1. **Pengecekan GPU (CUDA)**: Memastikan training berjalan cepat menggunakan GPU RTX 3050 Anda.
2. **Penyatuan Dataset**: Menggabungkan dataset Daun (8 kelas) dan Tongkol (2 kelas) menjadi 10 kelas.
3. **Melatih Model (Training)**: Melatih model YOLOv11 Classification.
4. **Konversi & Deployment**: Mengekspor model terbaik ke format TFLite dan menyalinnya langsung ke folder assets aplikasi Android.

--- 
## 🛠️ Step 1: Cek GPU & PyTorch
Jalankan cell di bawah untuk memastikan PyTorch Anda berjalan menggunakan GPU NVIDIA RTX 3050.

In [ ]:
import torch
import sys

print("=" * 60)
print(" SYSTEM GPU & PYTORCH STATUS")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA (GPU) Available: {cuda_available}")

if cuda_available:
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    device = "0"
    print("\n[OK] Laptop Anda siap untuk melakukan training menggunakan GPU RTX 3050! 🔥")
else:
    print("\n[WARNING] PyTorch saat ini berjalan menggunakan CPU.")
    print("Training YOLO menggunakan CPU akan berjalan sangat lambat.")
    print("Karena laptop Anda memiliki NVIDIA GeForce RTX 3050, silakan install CUDA PyTorch terlebih dahulu.")
    print("Caranya, jalankan perintah berikut di terminal VS Code Anda:")
    print("\n    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121\n")
    device = "cpu"

--- 
## 📁 Step 2: Penyatuan Dataset (Daun + Tongkol)
Cell ini akan menggabungkan dataset Daun (8 kelas) dan Tongkol (2 kelas) dari folder `Dataset_Revisi (+google)` ke folder baru bernama `Dataset_Merged` (total 10 kelas).

In [ ]:
import os
import shutil

daun_dir = r"C:\CornAI\DATA SEMUA\Dataset_Revisi (+google)\Daun"
tongkol_dir = r"C:\CornAI\DATA SEMUA\Dataset_Revisi (+google)\Tongkol"
merged_dir = r"C:\CornAI\DATA SEMUA\Dataset_Merged"

print(f"Source Daun: {daun_dir}")
print(f"Source Tongkol: {tongkol_dir}")
print(f"Target Merged: {merged_dir}")

# Bersihkan folder merged lama jika ada
if os.path.exists(merged_dir):
    print(f"Membersihkan folder lama: {merged_dir}...")
    shutil.rmtree(merged_dir)

splits_map = {
    "train": "train",
    "valid": "val",  # YOLO classification menggunakan nama folder 'val'
    "test": "test"
}

for src_split, dest_split in splits_map.items():
    dest_split_dir = os.path.join(merged_dir, dest_split)
    os.makedirs(dest_split_dir, exist_ok=True)
    
    # Copy Daun
    src_daun_split = os.path.join(daun_dir, src_split)
    if os.path.exists(src_daun_split):
        for class_name in os.listdir(src_daun_split):
            src_class_path = os.path.join(src_daun_split, class_name)
            dest_class_path = os.path.join(dest_split_dir, class_name)
            if os.path.isdir(src_class_path):
                shutil.copytree(src_class_path, dest_class_path)
                
    # Copy Tongkol
    src_tongkol_split = os.path.join(tongkol_dir, src_split)
    if os.path.exists(src_tongkol_split):
        for class_name in os.listdir(src_tongkol_split):
            src_class_path = os.path.join(src_tongkol_split, class_name)
            dest_class_path = os.path.join(dest_split_dir, class_name)
            if os.path.isdir(src_class_path):
                shutil.copytree(src_class_path, dest_class_path)

print("\n[SUCCESS] Dataset berhasil digabungkan!")
print("Daftar Kelas (Terurut Abjad):")
classes = sorted(os.listdir(os.path.join(merged_dir, "train")))
for idx, cls in enumerate(classes):
    print(f"  Index {idx}: {cls}")

--- 
## 🚀 Step 3: Mulai Training YOLOv11 Classification
Jalankan cell ini untuk mendownload bobot dasar YOLOv11 dan melatih model. Anda akan melihat visualisasi grafik latihan secara real-time di bawah cell ini nanti!

In [ ]:
from ultralytics import YOLO

# Memuat model dasar YOLOv11 Classification
print("Memuat pre-trained model yolo11n-cls.pt...")
model = YOLO("yolo11n-cls.pt")

# Konfigurasi training
epochs = 30  # Sesuaikan jumlah epoch (misal: 30, 50, atau 100)
imgsz = 224
batch_size = 32

print(f"Mulai melatih model pada device: {device}")
results = model.train(
    data=merged_dir,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch_size,
    device=device,
    workers=4,
    project="CornAI_Training",
    name="yolo11_classification"
)

best_weights = os.path.join("CornAI_Training", "yolo11_classification", "weights", "best.pt")
print(f"\n[SUCCESS] Latihan selesai! Model terbaik disimpan di: {best_weights}")

--- 
## 📦 Step 4: Konversi ke TFLite & Deploy ke Aplikasi Android
Cell ini akan mengonversi model `.pt` terbaik hasil training Anda ke format `.tflite` dan menyalinnya langsung ke folder `assets` aplikasi Android Anda agar siap dijalankan.

In [ ]:
if os.path.exists(best_weights):
    print("Memuat model terbaik untuk diekspor...")
    best_model = YOLO(best_weights)
    
    print("Mengekspor model ke format TFLite...")
    export_path = best_model.export(format="tflite", imgsz=224)
    
    # Cari file .tflite yang dihasilkan
    tflite_src = None
    if os.path.isdir(export_path):
        for f in os.listdir(export_path):
            if f.endswith(".tflite"):
                tflite_src = os.path.join(export_path, f)
                break
    elif os.path.isfile(export_path) and export_path.endswith(".tflite"):
        tflite_src = export_path
        
    if tflite_src and os.path.exists(tflite_src):
        # Copy ke Assets Android
        assets_dir = r"C:\CornAI\app\src\main\assets"
        os.makedirs(assets_dir, exist_ok=True)
        
        dest_tflite = os.path.join(assets_dir, "model_cornai.tflite")
        shutil.copy2(tflite_src, dest_tflite)
        print(f"[SUCCESS] Model TFLite berhasil dideploy ke: {dest_tflite}")
        
        # Update labels.txt
        dest_labels = os.path.join(assets_dir, "labels.txt")
        with open(dest_labels, "w") as lf:
            for cls in classes:
                lf.write(f"{cls}\n")
        print(f"[SUCCESS] File label berhasil diperbarui di: {dest_labels}")
        print("\n🔥 SELESAI! Silakan buka Android Studio, lalu Build & Run aplikasi Anda!")
    else:
        print("[ERROR] File .tflite hasil konversi tidak ditemukan.")
else:
    print("[ERROR] File best.pt tidak ditemukan. Silakan jalankan Step 3 terlebih dahulu.")